# 🦁 Savanna Acoustic Threat Detection — Kaggle GPU Training
**Repo:** https://github.com/mangaorphy/orpheus_reinforcement_learning_summative

> Enable: sidebar → Accelerator → **GPU T4 x2** and Internet → **On**

In [ ]:
# ── STEP 1: Clone your GitHub repo ──────────────────────────────────────────
import os, sys

REPO_URL  = 'https://github.com/mangaorphy/orpheus_reinforcement_learning_summative.git'
REPO_NAME = 'orpheus_reinforcement_learning_summative'
WORK_DIR  = f'/kaggle/working/{REPO_NAME}'

if os.path.exists(WORK_DIR):
    print('Repo exists — pulling latest...')
    os.system(f'cd {WORK_DIR} && git pull')
else:
    print('Cloning repo...')
    os.system(f'git clone {REPO_URL} {WORK_DIR}')

sys.path.insert(0, WORK_DIR)
os.chdir(WORK_DIR)
print(f'Working dir: {os.getcwd()}')
os.system('ls -la')

In [ ]:
# ── STEP 2: Install dependencies ────────────────────────────────────────────
import subprocess
for pkg in ['gymnasium==0.29.1','stable-baselines3==2.2.1','sb3-contrib==2.2.1','pygame==2.5.2','tqdm','rich','psutil']:
    r = subprocess.run([sys.executable,'-m','pip','install',pkg,'-q'], capture_output=True)
    print(f'  {pkg:<35} {"OK" if r.returncode==0 else "FAILED"}')

In [ ]:
# ── STEP 3: Setup headless display + output dirs ─────────────────────────────
os.environ['SDL_VIDEODRIVER'] = 'dummy'
os.environ['SDL_AUDIODRIVER'] = 'dummy'

MODELS_DIR  = f'{WORK_DIR}/models'
RESULTS_DIR = f'{WORK_DIR}/results'

for d in [f'{MODELS_DIR}/dqn', f'{MODELS_DIR}/pg/ppo', f'{MODELS_DIR}/pg/a2c', f'{MODELS_DIR}/pg/reinforce', RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

from environment.custom_env import SavannaAcousticEnv
env = SavannaAcousticEnv(render_mode=None, seed=42)
obs, _ = env.reset()
print(f'Obs shape: {obs.shape} | Actions: {env.action_space.n} | Threats: {len(env.threats)}')
env.close()
print('Environment OK.')

In [ ]:
# ── STEP 4: Memory management & shared utilities ─────────────────────────────
import gc, json, time
import numpy as np
import torch
import psutil
from stable_baselines3.common.monitor    import Monitor
from stable_baselines3.common.env_util   import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.callbacks  import EvalCallback, BaseCallback

TOTAL_TIMESTEPS = 100_000
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

def clear_memory(label=''):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    ram = psutil.virtual_memory()
    used = (ram.total - ram.available) / 1e9
    gpu  = f' | GPU: {torch.cuda.memory_allocated()/1e9:.2f}GB' if torch.cuda.is_available() else ''
    print(f'  [MEM] {label:<28} RAM: {used:.1f}/{ram.total/1e9:.1f}GB ({ram.percent:.0f}%){gpu}')
    if ram.percent > 85:
        print('  WARNING: RAM > 85%')

def check_memory():
    if psutil.virtual_memory().percent > 80:
        clear_memory('auto-clear')

def save_results(results, filename):
    path = os.path.join(RESULTS_DIR, filename)
    with open(path, 'w') as f:
        json.dump(results, f, indent=2)
    print(f'  Saved -> {path}')

def make_env_fn(seed=0):
    def _f():
        return Monitor(SavannaAcousticEnv(render_mode=None, seed=seed))
    return _f

class RewardLogger(BaseCallback):
    def __init__(self):
        super().__init__()
        self.episode_rewards = []
    def _on_step(self):
        for info in self.locals.get('infos', []):
            if 'episode' in info:
                self.episode_rewards.append(info['episode']['r'])
        return True

clear_memory('startup')
print('Utilities ready.')

In [ ]:
# ── STEP 5: Train DQN (10 runs × 100k steps) ─────────────────────────────────
from stable_baselines3 import DQN

DQN_CONFIGS = [
    {'name':'baseline',    'lr':1e-4,'buf':50000,  'bs':64,  'tau':1.0,  'gamma':0.99, 'exp_fr':0.20,'exp_fe':0.05,'arch':[128,128],    'notes':'SB3 defaults'},
    {'name':'high_lr',     'lr':5e-4,'buf':50000,  'bs':64,  'tau':1.0,  'gamma':0.99, 'exp_fr':0.20,'exp_fe':0.05,'arch':[128,128],    'notes':'High LR'},
    {'name':'deep_net',    'lr':1e-4,'buf':50000,  'bs':64,  'tau':1.0,  'gamma':0.99, 'exp_fr':0.20,'exp_fe':0.05,'arch':[256,256,128],'notes':'Deeper network'},
    {'name':'large_buf',   'lr':1e-4,'buf':200000, 'bs':128, 'tau':1.0,  'gamma':0.99, 'exp_fr':0.30,'exp_fe':0.05,'arch':[128,128],    'notes':'Large buffer'},
    {'name':'low_gamma',   'lr':1e-4,'buf':50000,  'bs':64,  'tau':1.0,  'gamma':0.90, 'exp_fr':0.20,'exp_fe':0.05,'arch':[128,128],    'notes':'Low gamma'},
    {'name':'soft_update', 'lr':1e-4,'buf':50000,  'bs':64,  'tau':0.01, 'gamma':0.99, 'exp_fr':0.20,'exp_fe':0.05,'arch':[128,128],    'notes':'Polyak update'},
    {'name':'long_explore','lr':1e-4,'buf':100000, 'bs':64,  'tau':1.0,  'gamma':0.99, 'exp_fr':0.50,'exp_fe':0.10,'arch':[128,128],    'notes':'Long exploration'},
    {'name':'large_batch', 'lr':2e-4,'buf':100000, 'bs':256, 'tau':1.0,  'gamma':0.99, 'exp_fr':0.20,'exp_fe':0.05,'arch':[256,256],    'notes':'Large batch'},
    {'name':'fast_target', 'lr':1e-4,'buf':50000,  'bs':64,  'tau':1.0,  'gamma':0.99, 'exp_fr':0.15,'exp_fe':0.02,'arch':[128,128],    'notes':'Fast target update'},
    {'name':'best_tuned',  'lr':2e-4,'buf':100000, 'bs':128, 'tau':0.05, 'gamma':0.995,'exp_fr':0.30,'exp_fe':0.03,'arch':[256,128,64], 'notes':'Best tuned'},
]

dqn_results = []
for i, cfg in enumerate(DQN_CONFIGS, 1):
    check_memory()
    print(f'\n--- DQN Run {i}/10: {cfg["name"]} | {cfg["notes"]} ---')
    train_env = make_vec_env(make_env_fn(i*100), n_envs=1)
    eval_env  = Monitor(SavannaAcousticEnv(render_mode=None, seed=999))
    logger    = RewardLogger()
    eval_cb   = EvalCallback(eval_env, best_model_save_path=f'{MODELS_DIR}/dqn/{cfg["name"]}', eval_freq=5000, n_eval_episodes=5, verbose=0)
    model = DQN('MlpPolicy', train_env, learning_rate=cfg['lr'], buffer_size=cfg['buf'], batch_size=cfg['bs'], tau=cfg['tau'], gamma=cfg['gamma'], exploration_fraction=cfg['exp_fr'], exploration_final_eps=cfg['exp_fe'], learning_starts=1000, train_freq=4, target_update_interval=1000, policy_kwargs={'net_arch':cfg['arch']}, verbose=1, seed=i*7)
    t0 = time.time()
    model.learn(TOTAL_TIMESTEPS, callback=[logger, eval_cb], progress_bar=True)
    elapsed = time.time() - t0
    mean_r, std_r = evaluate_policy(model, eval_env, n_eval_episodes=10)
    os.makedirs(f'{MODELS_DIR}/dqn/{cfg["name"]}', exist_ok=True)
    model.save(f'{MODELS_DIR}/dqn/{cfg["name"]}/final_model')
    res = {'algorithm':'DQN','run_id':i,'config_name':cfg['name'],'notes':cfg['notes'],'mean_reward':float(mean_r),'std_reward':float(std_r),'training_time_s':float(elapsed),'hyperparameters':cfg,'episode_rewards':logger.episode_rewards[-50:]}
    dqn_results.append(res)
    print(f'  Result: {mean_r:.2f} +/- {std_r:.2f} | Time: {elapsed/60:.1f} min')
    del model, train_env, eval_env, logger
    clear_memory(f'DQN {i}/10 done')

save_results(dqn_results, 'dqn_all_results.json')
dqn_results.sort(key=lambda r: r['mean_reward'], reverse=True)
print(f'\nBest DQN: {dqn_results[0]["config_name"]} ({dqn_results[0]["mean_reward"]:.2f})')

In [ ]:
# ── STEP 6: Train PPO (10 runs × 100k steps) ─────────────────────────────────
from stable_baselines3 import PPO
clear_memory('before PPO')

PPO_CONFIGS = [
    {'name':'ppo_baseline','lr':3e-4,'n_steps':2048,'bs':64, 'epochs':10,'gamma':0.99, 'gae':0.95,'clip':0.2,'ent':0.00,'vf':0.5,'arch':[128,128],   'notes':'SB3 defaults'},
    {'name':'ppo_sm_steps','lr':3e-4,'n_steps':512, 'bs':64, 'epochs':10,'gamma':0.99, 'gae':0.95,'clip':0.2,'ent':0.00,'vf':0.5,'arch':[128,128],   'notes':'Small rollout'},
    {'name':'ppo_lg_steps','lr':3e-4,'n_steps':4096,'bs':128,'epochs':10,'gamma':0.99, 'gae':0.95,'clip':0.2,'ent':0.00,'vf':0.5,'arch':[256,256],   'notes':'Large rollout'},
    {'name':'ppo_hi_clip', 'lr':3e-4,'n_steps':2048,'bs':64, 'epochs':10,'gamma':0.99, 'gae':0.95,'clip':0.4,'ent':0.00,'vf':0.5,'arch':[128,128],   'notes':'High clip'},
    {'name':'ppo_lo_clip', 'lr':3e-4,'n_steps':2048,'bs':64, 'epochs':10,'gamma':0.99, 'gae':0.95,'clip':0.1,'ent':0.00,'vf':0.5,'arch':[128,128],   'notes':'Low clip'},
    {'name':'ppo_entropy', 'lr':3e-4,'n_steps':2048,'bs':64, 'epochs':10,'gamma':0.99, 'gae':0.95,'clip':0.2,'ent':0.01,'vf':0.5,'arch':[128,128],   'notes':'Entropy bonus'},
    {'name':'ppo_high_vf', 'lr':3e-4,'n_steps':2048,'bs':64, 'epochs':10,'gamma':0.99, 'gae':0.95,'clip':0.2,'ent':0.00,'vf':1.0,'arch':[128,128],   'notes':'High VF coeff'},
    {'name':'ppo_low_gae', 'lr':3e-4,'n_steps':2048,'bs':64, 'epochs':10,'gamma':0.99, 'gae':0.80,'clip':0.2,'ent':0.00,'vf':0.5,'arch':[128,128],   'notes':'Low GAE lambda'},
    {'name':'ppo_more_ep', 'lr':2e-4,'n_steps':2048,'bs':64, 'epochs':20,'gamma':0.99, 'gae':0.95,'clip':0.2,'ent':0.01,'vf':0.5,'arch':[256,128],   'notes':'More epochs'},
    {'name':'ppo_best',    'lr':2e-4,'n_steps':2048,'bs':128,'epochs':15,'gamma':0.995,'gae':0.95,'clip':0.2,'ent':0.01,'vf':0.5,'arch':[256,128,64],'notes':'Best tuned'},
]

ppo_results = []
for i, cfg in enumerate(PPO_CONFIGS, 1):
    check_memory()
    print(f'\n--- PPO Run {i}/10: {cfg["name"]} | {cfg["notes"]} ---')
    train_env = make_vec_env(make_env_fn(i*100), n_envs=1)
    eval_env  = Monitor(SavannaAcousticEnv(render_mode=None, seed=999))
    logger    = RewardLogger()
    eval_cb   = EvalCallback(eval_env, best_model_save_path=f'{MODELS_DIR}/pg/ppo/{cfg["name"]}', eval_freq=5000, n_eval_episodes=5, verbose=0)
    model = PPO('MlpPolicy', train_env, learning_rate=cfg['lr'], n_steps=cfg['n_steps'], batch_size=cfg['bs'], n_epochs=cfg['epochs'], gamma=cfg['gamma'], gae_lambda=cfg['gae'], clip_range=cfg['clip'], ent_coef=cfg['ent'], vf_coef=cfg['vf'], policy_kwargs={'net_arch':cfg['arch']}, verbose=1, seed=i*7)
    t0 = time.time()
    model.learn(TOTAL_TIMESTEPS, callback=[logger, eval_cb], progress_bar=True)
    elapsed = time.time() - t0
    mean_r, std_r = evaluate_policy(model, eval_env, n_eval_episodes=10)
    os.makedirs(f'{MODELS_DIR}/pg/ppo/{cfg["name"]}', exist_ok=True)
    model.save(f'{MODELS_DIR}/pg/ppo/{cfg["name"]}/final_model')
    res = {'algorithm':'PPO','run_id':i,'config_name':cfg['name'],'notes':cfg['notes'],'mean_reward':float(mean_r),'std_reward':float(std_r),'training_time_s':float(elapsed),'hyperparameters':cfg,'episode_rewards':logger.episode_rewards[-50:]}
    ppo_results.append(res)
    print(f'  Result: {mean_r:.2f} +/- {std_r:.2f} | Time: {elapsed/60:.1f} min')
    del model, train_env, eval_env, logger
    clear_memory(f'PPO {i}/10 done')

save_results(ppo_results, 'ppo_all_results.json')
ppo_results.sort(key=lambda r: r['mean_reward'], reverse=True)
print(f'\nBest PPO: {ppo_results[0]["config_name"]} ({ppo_results[0]["mean_reward"]:.2f})')

In [ ]:
# ── STEP 7: Train A2C (10 runs × 100k steps) ─────────────────────────────────
from stable_baselines3 import A2C
clear_memory('before A2C')

A2C_CONFIGS = [
    {'name':'a2c_baseline','lr':7e-4,'n_steps':5, 'gamma':0.99, 'gae':1.0, 'ent':0.00,'vf':0.5,'gnorm':0.5,'arch':[64,64],      'notes':'SB3 defaults'},
    {'name':'a2c_steps20', 'lr':7e-4,'n_steps':20,'gamma':0.99, 'gae':1.0, 'ent':0.00,'vf':0.5,'gnorm':0.5,'arch':[128,128],    'notes':'Longer rollouts'},
    {'name':'a2c_high_lr', 'lr':1e-3,'n_steps':5, 'gamma':0.99, 'gae':1.0, 'ent':0.00,'vf':0.5,'gnorm':0.5,'arch':[128,128],    'notes':'High LR'},
    {'name':'a2c_gae',     'lr':7e-4,'n_steps':10,'gamma':0.99, 'gae':0.95,'ent':0.00,'vf':0.5,'gnorm':0.5,'arch':[128,128],    'notes':'GAE estimation'},
    {'name':'a2c_entropy', 'lr':7e-4,'n_steps':5, 'gamma':0.99, 'gae':1.0, 'ent':0.01,'vf':0.5,'gnorm':0.5,'arch':[128,128],    'notes':'Entropy bonus'},
    {'name':'a2c_high_vf', 'lr':7e-4,'n_steps':5, 'gamma':0.99, 'gae':1.0, 'ent':0.00,'vf':1.0,'gnorm':0.5,'arch':[128,128],    'notes':'High VF coeff'},
    {'name':'a2c_low_gam', 'lr':7e-4,'n_steps':5, 'gamma':0.90, 'gae':1.0, 'ent':0.00,'vf':0.5,'gnorm':0.5,'arch':[128,128],    'notes':'Low gamma'},
    {'name':'a2c_deep',    'lr':5e-4,'n_steps':10,'gamma':0.99, 'gae':0.95,'ent':0.01,'vf':0.5,'gnorm':0.5,'arch':[256,256,128],'notes':'Deep network'},
    {'name':'a2c_gradclip','lr':7e-4,'n_steps':5, 'gamma':0.99, 'gae':1.0, 'ent':0.00,'vf':0.5,'gnorm':0.1,'arch':[128,128],    'notes':'Tight grad clip'},
    {'name':'a2c_best',    'lr':5e-4,'n_steps':20,'gamma':0.995,'gae':0.95,'ent':0.01,'vf':0.5,'gnorm':0.5,'arch':[256,128],    'notes':'Best tuned'},
]

a2c_results = []
for i, cfg in enumerate(A2C_CONFIGS, 1):
    check_memory()
    print(f'\n--- A2C Run {i}/10: {cfg["name"]} | {cfg["notes"]} ---')
    train_env = make_vec_env(make_env_fn(i*100), n_envs=1)
    eval_env  = Monitor(SavannaAcousticEnv(render_mode=None, seed=999))
    logger    = RewardLogger()
    eval_cb   = EvalCallback(eval_env, best_model_save_path=f'{MODELS_DIR}/pg/a2c/{cfg["name"]}', eval_freq=5000, n_eval_episodes=5, verbose=0)
    model = A2C('MlpPolicy', train_env, learning_rate=cfg['lr'], n_steps=cfg['n_steps'], gamma=cfg['gamma'], gae_lambda=cfg['gae'], ent_coef=cfg['ent'], vf_coef=cfg['vf'], max_grad_norm=cfg['gnorm'], policy_kwargs={'net_arch':cfg['arch']}, verbose=1, seed=i*7)
    t0 = time.time()
    model.learn(TOTAL_TIMESTEPS, callback=[logger, eval_cb], progress_bar=True)
    elapsed = time.time() - t0
    mean_r, std_r = evaluate_policy(model, eval_env, n_eval_episodes=10)
    os.makedirs(f'{MODELS_DIR}/pg/a2c/{cfg["name"]}', exist_ok=True)
    model.save(f'{MODELS_DIR}/pg/a2c/{cfg["name"]}/final_model')
    res = {'algorithm':'A2C','run_id':i,'config_name':cfg['name'],'notes':cfg['notes'],'mean_reward':float(mean_r),'std_reward':float(std_r),'training_time_s':float(elapsed),'hyperparameters':cfg,'episode_rewards':logger.episode_rewards[-50:]}
    a2c_results.append(res)
    print(f'  Result: {mean_r:.2f} +/- {std_r:.2f} | Time: {elapsed/60:.1f} min')
    del model, train_env, eval_env, logger
    clear_memory(f'A2C {i}/10 done')

save_results(a2c_results, 'a2c_all_results.json')
a2c_results.sort(key=lambda r: r['mean_reward'], reverse=True)
print(f'\nBest A2C: {a2c_results[0]["config_name"]} ({a2c_results[0]["mean_reward"]:.2f})')

In [ ]:
# ── STEP 8: Train REINFORCE (10 runs × 100k steps) ───────────────────────────
import torch.nn as nn
clear_memory('before REINFORCE')

class REINFORCEPolicy(nn.Module):
    def __init__(self, obs_dim, action_dim, hidden=(128,128)):
        super().__init__()
        layers, in_d = [], obs_dim
        for h in hidden:
            layers += [nn.Linear(in_d,h), nn.ReLU()]
            in_d = h
        layers.append(nn.Linear(in_d, action_dim))
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x)
    def get_action(self, obs_t):
        dist = torch.distributions.Categorical(logits=self.forward(obs_t))
        a = dist.sample()
        return a, dist.log_prob(a)
    def evaluate(self, obs_t, actions):
        dist = torch.distributions.Categorical(logits=self.forward(obs_t))
        return dist.log_prob(actions), dist.entropy()

def train_reinforce(cfg, run_id, total_timesteps):
    env   = Monitor(SavannaAcousticEnv(render_mode=None, seed=run_id*100))
    e_env = Monitor(SavannaAcousticEnv(render_mode=None, seed=999))
    policy = REINFORCEPolicy(env.observation_space.shape[0], env.action_space.n, cfg['arch'])
    opt    = torch.optim.Adam(policy.parameters(), lr=cfg['lr'])
    ep_rewards, total_steps = [], 0
    while total_steps < total_timesteps:
        obs, _ = env.reset()
        states, actions, rewards, log_probs = [], [], [], []
        done = False
        while not done:
            obs_t = torch.FloatTensor(obs).unsqueeze(0)
            a, lp = policy.get_action(obs_t)
            obs, r, term, trunc, _ = env.step(a.item())
            done = term or trunc
            states.append(obs_t); actions.append(a); rewards.append(r); log_probs.append(lp)
            total_steps += 1
        ep_rewards.append(sum(rewards))
        G, returns = 0.0, []
        for r in reversed(rewards):
            G = r + cfg['gamma'] * G
            returns.insert(0, G)
        ret_t = torch.FloatTensor(returns)
        if cfg['baseline']: ret_t = (ret_t - ret_t.mean()) / (ret_t.std() + 1e-8)
        lp_t = torch.stack(log_probs)
        _, entropy = policy.evaluate(torch.cat(states), torch.stack(actions))
        loss = -(lp_t * ret_t).mean() - cfg['ent_coef'] * entropy.mean()
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 0.5)
        opt.step()
        if len(ep_rewards) % 20 == 0:
            print(f'    Step {total_steps:>7} | Ep {len(ep_rewards):>4} | Avg(20): {np.mean(ep_rewards[-20:]):.2f}')
    eval_rewards = []
    for _ in range(10):
        obs, _ = e_env.reset()
        ep_r, done = 0.0, False
        while not done:
            with torch.no_grad(): a, _ = policy.get_action(torch.FloatTensor(obs).unsqueeze(0))
            obs, r, term, trunc, _ = e_env.step(a.item())
            ep_r += r; done = term or trunc
        eval_rewards.append(ep_r)
    save_dir = f'{MODELS_DIR}/pg/reinforce/{cfg["name"]}'
    os.makedirs(save_dir, exist_ok=True)
    torch.save({'policy_state':policy.state_dict(),'config':cfg}, f'{save_dir}/final_model.pt')
    env.close(); e_env.close(); del policy, opt
    return float(np.mean(eval_rewards)), float(np.std(eval_rewards)), ep_rewards

RF_CONFIGS = [
    {'name':'rf_baseline',   'lr':1e-3,'gamma':0.99, 'baseline':True, 'ent_coef':0.01,'arch':(128,128),    'notes':'Standard + baseline'},
    {'name':'rf_no_baseline','lr':1e-3,'gamma':0.99, 'baseline':False,'ent_coef':0.01,'arch':(128,128),    'notes':'No baseline'},
    {'name':'rf_high_lr',    'lr':5e-3,'gamma':0.99, 'baseline':True, 'ent_coef':0.01,'arch':(128,128),    'notes':'High LR'},
    {'name':'rf_low_lr',     'lr':1e-4,'gamma':0.99, 'baseline':True, 'ent_coef':0.01,'arch':(128,128),    'notes':'Low LR'},
    {'name':'rf_low_gamma',  'lr':1e-3,'gamma':0.90, 'baseline':True, 'ent_coef':0.01,'arch':(128,128),    'notes':'Low gamma'},
    {'name':'rf_deep',       'lr':1e-3,'gamma':0.99, 'baseline':True, 'ent_coef':0.01,'arch':(256,256,128),'notes':'Deep network'},
    {'name':'rf_hi_entropy', 'lr':1e-3,'gamma':0.99, 'baseline':True, 'ent_coef':0.05,'arch':(128,128),    'notes':'High entropy'},
    {'name':'rf_no_entropy', 'lr':1e-3,'gamma':0.99, 'baseline':True, 'ent_coef':0.00,'arch':(128,128),    'notes':'No entropy'},
    {'name':'rf_wide',       'lr':2e-3,'gamma':0.99, 'baseline':True, 'ent_coef':0.01,'arch':(512,256),    'notes':'Wide network'},
    {'name':'rf_best',       'lr':2e-3,'gamma':0.995,'baseline':True, 'ent_coef':0.02,'arch':(256,128),    'notes':'Best tuned'},
]

rf_results = []
for i, cfg in enumerate(RF_CONFIGS, 1):
    check_memory()
    print(f'\n--- REINFORCE Run {i}/10: {cfg["name"]} | {cfg["notes"]} ---')
    t0 = time.time()
    mean_r, std_r, ep_rews = train_reinforce(cfg, i, TOTAL_TIMESTEPS)
    elapsed = time.time() - t0
    res = {'algorithm':'REINFORCE','run_id':i,'config_name':cfg['name'],'notes':cfg['notes'],'mean_reward':mean_r,'std_reward':std_r,'training_time_s':elapsed,'hyperparameters':cfg,'episode_rewards':ep_rews[-50:]}
    rf_results.append(res)
    print(f'  Result: {mean_r:.2f} +/- {std_r:.2f} | Time: {elapsed/60:.1f} min')
    clear_memory(f'RF {i}/10 done')

save_results(rf_results, 'reinforce_all_results.json')
rf_results.sort(key=lambda r: r['mean_reward'], reverse=True)
print(f'\nBest REINFORCE: {rf_results[0]["config_name"]} ({rf_results[0]["mean_reward"]:.2f})')

In [ ]:
# ── STEP 9: Cross-algorithm comparison chart ──────────────────────────────────
import matplotlib.pyplot as plt

all_bests = [dqn_results[0], ppo_results[0], a2c_results[0], rf_results[0]]
print('='*65)
print('  CROSS-ALGORITHM COMPARISON')
print('='*65)
print(f'  {"Algorithm":<12} {"Best Config":<22} {"Mean Reward":>12} {"Std":>8}')
print('  '+'-'*56)
for r in all_bests:
    print(f'  {r["algorithm"]:<12} {r["config_name"]:<22} {r["mean_reward"]:>12.2f} {r["std_reward"]:>8.2f}')
print('='*65)
best = max(all_bests, key=lambda r: r['mean_reward'])
print(f'\n  Overall best: {best["algorithm"]} — {best["config_name"]} ({best["mean_reward"]:.2f})')

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.patch.set_facecolor('#0F172A')
algos   = [r['algorithm'] for r in all_bests]
rewards = [r['mean_reward'] for r in all_bests]
stds    = [r['std_reward'] for r in all_bests]
times   = [r['training_time_s']/60 for r in all_bests]
colors  = ['#3B82F6','#10B981','#F59E0B','#8B5CF6']
for ax, vals, yerr, title, ylabel in [
    (axes[0], rewards, stds,  'Mean reward per algorithm', 'Mean Reward'),
    (axes[1], times,   None,  'Training time (minutes)',   'Minutes')
]:
    ax.set_facecolor('#1E293B')
    bars = ax.bar(algos, vals, yerr=yerr, color=colors, alpha=0.85, edgecolor='white', linewidth=0.8, capsize=6 if yerr else 0)
    ax.set_title(title, color='white', pad=10)
    ax.set_ylabel(ylabel, color='white')
    ax.tick_params(colors='white')
    for spine in ax.spines.values(): spine.set_color('#334155')
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.5, f'{v:.1f}', ha='center', color='white', fontsize=9)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}/algorithm_comparison.png', dpi=150, bbox_inches='tight', facecolor='#0F172A')
plt.show()
print('Chart saved.')

In [ ]:
# ── STEP 10: Push results back to GitHub ──────────────────────────────────────
os.chdir(WORK_DIR)
os.system('git config user.email "kaggle@training.com"')
os.system('git config user.name "Kaggle GPU Training"')
os.system('git add results/ models/')
os.system('git commit -m "Add trained models and results — Kaggle GPU 100k steps"')

# Get your token: https://github.com/settings/tokens → New token (classic) → tick repo
GITHUB_TOKEN = 'YOUR_GITHUB_TOKEN_HERE'  # <── paste token here
REPO = 'mangaorphy/orpheus_reinforcement_learning_summative'
push_url = f'https://{GITHUB_TOKEN}@github.com/{REPO}.git'
result = os.system(f'git push {push_url} main')
if result == 0:
    print(f'Pushed! View at: https://github.com/{REPO}')
else:
    print('Push failed. Use the zip download below instead.')

In [ ]:
# ── STEP 11: Zip for download (if GitHub push fails) ─────────────────────────
import shutil
shutil.make_archive('/kaggle/working/models_trained',  'zip', f'{WORK_DIR}/models')
shutil.make_archive('/kaggle/working/results_trained', 'zip', f'{WORK_DIR}/results')
print('Download these from the Kaggle Output tab:')
print('  /kaggle/working/models_trained.zip')
print('  /kaggle/working/results_trained.zip')